# 📘 The AI Engineer's LLM Workbook

**14 Chapters · 14 Google Colab Notebooks · Beginner to Production**

---

*© 2026 JAWNVION LLC — www.jawnvion.com — peter@jawnvion.com*

*Licensed for individual use. Do not redistribute.*

---

## What's Inside

| # | Chapter |
|---|---------|
| 01 | AI Fundamentals & Problem Framing |
| 02 | Data Science Toolkit (NumPy, Pandas, Matplotlib) |
| 03 | Neural Networks from Scratch |
| 04 | Transformers Architecture Deep Dive |
| 05 | HuggingFace & Pre-Trained Models |
| 06 | QLoRA Fine-Tuning |
| 07 | DPO Alignment Training |
| 08 | Retrieval-Augmented Generation (RAG) |
| 09 | Model Evaluation & Benchmarking |
| 10 | FastAPI Deployment |
| 11 | Monitoring & Observability |
| 12 | Security for AI Systems |
| 13 | Cost Optimization & Quantization |
| 14 | Capstone: End-to-End LLM Project |

---

> **How to use:** Click **Runtime → Run All** in Google Colab, or run cells one at a time.
> Each chapter builds on the last — complete them in order for best results.

---


# Chapter 7: DPO Fine-Tuning (Direct Preference Optimization)
**JAWNVION LLC — AI Training Workbook**

DPO teaches a model to *prefer* good responses over bad ones — without a separate reward model.
We build on the QLoRA adapter from Chapter 6 and apply DPO using 500 human-preference pairs.

**What you'll learn:**
- How DPO differs from SFT and RLHF
- How to format a preference dataset (prompt / chosen / rejected)
- How to run `DPOTrainer` from TRL on a T4 GPU
- How to measure the KL-divergence reward signal during training

In [ ]:
# — Cell 1: GPU Check —————————————————————————————————
import subprocess, sys
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                        '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode == 0:
    print('✓  GPU detected:', result.stdout.strip())
else:
    print('✗  No GPU found — go to Runtime → Change runtime type → T4 GPU')
    sys.exit(1)

In [ ]:
# — Cell 2: Install Packages ——————————————————————————
# Restart runtime after this cell if prompted
!pip install -q "tokenizers>=0.22,<0.24"
!pip install -q -U transformers trl peft bitsandbytes accelerate datasets
print('✓  Packages installed — if you see a RESTART button, click it then run all again')

In [ ]:
# — Cell 3: Imports & Configuration ———————————————————
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import DPOTrainer, DPOConfig
from datasets import load_dataset
import time, os

# ── Constants ─────────────────────────────────────────
BASE_MODEL     = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
SFT_ADAPTER    = "/content/qlora-tinyllama"   # Chapter 6 output; if missing we start from base
OUTPUT_DIR     = "/content/dpo-tinyllama"
NUM_DPO_EXAMPLES = 500
MAX_LEN        = 512
BETA           = 0.1    # KL-penalty coefficient (0.1 = standard)

print('✓  Config ready')
print(f'   Base model : {BASE_MODEL}')
print(f'   DPO beta   : {BETA}')
print(f'   Examples   : {NUM_DPO_EXAMPLES}')

In [ ]:
# — Cell 4: Load Model in 4-bit ————————————————————————
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"   # DPO requires left-padding

print("Loading base model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False

# Load Chapter 6 LoRA adapter if it exists; otherwise train from base
if os.path.isdir(SFT_ADAPTER):
    print(f"Loading Chapter 6 LoRA adapter from {SFT_ADAPTER}...")
    model = PeftModel.from_pretrained(model, SFT_ADAPTER, is_trainable=True)
    print("✓  SFT adapter loaded — DPO will refine the Chapter 6 model")
else:
    print("Chapter 6 adapter not found — applying fresh LoRA to base model")
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                        "gate_proj","up_proj","down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_config)
    print("✓  Fresh LoRA applied to base model")

model.print_trainable_parameters()
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# — Cell 5: Load & Format Preference Dataset ————————————
# Anthropic HH-RLHF: human-preference pairs (chosen vs rejected responses)
# Each row has a full conversation string; we split at the last Human/Assistant turn.

print("Loading Anthropic HH-RLHF dataset...")
raw = load_dataset("Anthropic/hh-rlhf", split="train")
raw = raw.shuffle(seed=42).select(range(NUM_DPO_EXAMPLES))

def extract_prompt_chosen_rejected(example):
    """
    HH-RLHF stores the full conversation in 'chosen' and 'rejected' strings.
    We extract the shared prompt (everything up to the last Assistant: turn)
    and the two different responses.
    """
    chosen_text   = example["chosen"]
    rejected_text = example["rejected"]

    # Find where the final Assistant response begins
    split_marker = "\n\nAssistant:"
    last_idx = chosen_text.rfind(split_marker)
    if last_idx == -1:
        return None

    prompt   = chosen_text[:last_idx + len(split_marker)].strip()
    chosen   = chosen_text[last_idx + len(split_marker):].strip()
    rejected = rejected_text[rejected_text.rfind(split_marker) + len(split_marker):].strip()

    return {"prompt": prompt, "chosen": chosen, "rejected": rejected}

print("Formatting preference pairs...")
dataset = raw.map(extract_prompt_chosen_rejected, remove_columns=raw.column_names)
dataset = dataset.filter(lambda x: x is not None and
                         len(x["prompt"]) > 10 and
                         len(x["chosen"]) > 5 and
                         len(x["rejected"]) > 5)

print(f"✓  Dataset ready: {len(dataset)} preference pairs")
print()
print("Sample row:")
print(f"  PROMPT   : {dataset[0]['prompt'][:120]}...")
print(f"  CHOSEN   : {dataset[0]['chosen'][:80]}...")
print(f"  REJECTED : {dataset[0]['rejected'][:80]}...")

In [ ]:
# — Cell 6: DPO Training Configuration ————————————————
from trl import DPOConfig

dpo_config = DPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,    # effective batch = 16
    optim="paged_adamw_8bit",
    learning_rate=5e-5,               # lower than SFT — DPO is fine-grained
    weight_decay=0.001,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    fp16=False,
    bf16=False,
    max_grad_norm=0.3,
    gradient_checkpointing=True,
    beta=BETA,                        # KL-divergence penalty weight
    max_length=MAX_LEN,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    report_to="none",
)

total_steps = (NUM_DPO_EXAMPLES // (2 * 8)) * 1
print("✓  DPOConfig ready")
print(f"   Beta (KL penalty) : {dpo_config.beta}")
print(f"   Effective batch   : {2 * 8}")
print(f"   ~Total steps      : {total_steps}")
print()
print("What beta does:")
print("  β=0.1 → gentle alignment, preserves SFT knowledge")
print("  β=0.5 → stronger alignment, may reduce fluency")

In [ ]:
# — Cell 7: Build DPO Trainer ——————————————————————————
from trl import DPOTrainer

# ref_model=None tells TRL to use an implicit reference (frozen copy of
# the initial weights).  This saves ~7 GB vs loading a second model.
trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("✓  DPOTrainer ready")
print("   Reference model : implicit (frozen initial weights — saves GPU memory)")
print("   Policy model    : Chapter 6 LoRA adapter" if os.path.isdir(SFT_ADAPTER)
      else "   Policy model    : fresh LoRA on TinyLlama base")

In [ ]:
# — Cell 8: Run DPO Training ——————————————————————————
print("=" * 60)
print("  Starting DPO Fine-Tuning")
print(f"  Model  : {BASE_MODEL}")
print(f"  Data   : {len(dataset)} preference pairs")
print(f"  Beta   : {BETA}")
print("=" * 60)
print()
print("Training loss will print every 10 steps.")
print("Rewards/chosen and rewards/rejected will also appear.")
print("A POSITIVE rewards margin means the model prefers chosen over rejected.")
print()

start_time = time.time()
train_result = trainer.train()
elapsed = time.time() - start_time

print()
print("=" * 60)
print(f"✓  DPO Training complete in {elapsed/60:.1f} minutes")
print(f"   Final loss   : {train_result.training_loss:.4f}")
print("=" * 60)

In [ ]:
# — Cell 9: Save DPO Adapter ————————————————————————————
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Also zip for easy download
import shutil
shutil.make_archive("/content/dpo-adapter-backup", "zip", OUTPUT_DIR)

print(f"✓  DPO adapter saved to {OUTPUT_DIR}")
print("   dpo-adapter-backup.zip also created — download from Files panel")

In [ ]:
# — Cell 10: Before / After Comparison ————————————————
# Tests whether DPO shifted the model toward more helpful, less harmful responses

TEST_PROMPT = (
    "Human: What is the best way to learn a new programming language?\n\n"
    "Assistant:"
)

def generate(mdl, prompt, max_new=120):
    inputs = tokenizer(prompt, return_tensors="pt").to(mdl.device)
    with torch.no_grad():
        out = mdl.generate(
            **inputs,
            max_new_tokens=max_new,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.3,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                            skip_special_tokens=True)

print("PROMPT:", TEST_PROMPT)
print()
print("─" * 60)
print("DPO-TUNED RESPONSE:")
print(generate(trainer.model, TEST_PROMPT))
print("─" * 60)
print()
print("Note: DPO aligns tone and helpfulness — not just factual accuracy.")
print("Compare this output with the SFT-only response from Chapter 6.")

## Chapter 7 Complete ✓

**What happened:**
- Loaded the Chapter 6 QLoRA adapter (or applied fresh LoRA if not available)
- Formatted 500 human-preference pairs from Anthropic HH-RLHF
- Trained with DPO (β=0.1) — no reward model needed
- Saved the DPO-aligned adapter to `/content/dpo-tinyllama`

**Key DPO concepts demonstrated:**
- `beta` controls the KL-divergence penalty (how far the policy can drift from reference)
- `rewards/chosen` should be higher than `rewards/rejected` after training
- `rewards/margin` = chosen − rejected; a positive margin = alignment is working
- Implicit reference model saves GPU memory vs loading two full models

**Next: Chapter 8 — RAG (Retrieval-Augmented Generation)**